# Single-clip and awake–sleep response associations

## Objective

This notebook asks a narrower question than reproducing the complete source paper: **are the published EEG quantities associated with immediate or sustained response when we calculate them from individual clips?** It then asks whether the published POST score remains associated with response when one awake clip and one sleep clip are combined.

A result counts as statistically significant here when its patient-level permutation P-value remains below 0.05 after correcting for the eight displayed comparisons within that response endpoint.

## What is being compared

The cohort contains 50 patients. Immediate response has 32 responders and 18 non-responders; sustained response has 28 responders and 22 non-responders. Each relevant condition/state cell contains two clips per patient.

The single-clip rows use one EEG clip at a time. The POST awake+sleep row forms all four possible combinations of one of a patient's two awake clips with one of that patient's two sleep clips. We also show the original patient-averaged construction as a reference. There is no PRE sleep+awake score because the published PRE formula uses awake EEG only.

The source paper selected these quantities for sustained response. Applying them to immediate response is therefore a new exploratory analysis. The local DFA and PLI distributions closely reproduce the source paper, and the published PRE and POST score formulas are used without changing their coefficients.

In [1]:
from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

SEED = 20260910
PERMUTATIONS = 1_000_000
REPO = Path(os.environ.get("IESSEEG_BASELINES_REPO", Path.cwd())).resolve()
METADATA_CSV = Path(os.environ["IESSEEG_METADATA_CSV"])
RAW_FEATURES_DIR = Path(os.environ["IESSEEG_RAJARAMAN_RAW_FEATURES_DIR"])

if not METADATA_CSV.is_file():
    raise FileNotFoundError("IESSEEG_METADATA_CSV does not point to a file")
if not RAW_FEATURES_DIR.is_dir():
    raise NotADirectoryError("IESSEEG_RAJARAMAN_RAW_FEATURES_DIR does not point to a directory")

analysis_dir = REPO / "analysis" / "response_features"
sys.path.insert(0, str(analysis_dir))
run_analysis = importlib.import_module("analyze_response_feature_associations").run_analysis

print(f"Configuration loaded: {PERMUTATIONS:,} patient-level permutations; seed {SEED}.")

Configuration loaded: 1,000,000 patient-level permutations; seed 20260910.


## Statistical test

The null hypothesis is that the EEG quantity and the response label are unrelated across patients. AUROC measures how consistently the quantity orders responder and non-responder observations. `Separation AUROC` is reported above 0.5 in the more informative direction; the adjacent direction column says whether responders have higher or lower values.

Although a single-clip row contains 100 values, it does **not** contain 100 independent patients. During every permutation, the response labels are shuffled across the 50 patients and all values from one patient move together. The test is two-sided. Holm correction accounts for the eight displayed comparisons within immediate response and, separately, within sustained response.

In [2]:
results = run_analysis(
    metadata_csv=METADATA_CSV,
    raw_features_dir=RAW_FEATURES_DIR,
    n_permutations=PERMUTATIONS,
    seed=SEED,
)

output_path = REPO / "local_results" / "rajaraman2024" / "clip_and_state_response_associations.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(output_path, index=False)
print(f"Completed {len(results)} comparisons. Aggregate table saved under local_results/.")

Completed 16 comparisons. Aggregate table saved under local_results/.


In [3]:
def readable_table(frame: pd.DataFrame) -> pd.io.formats.style.Styler:
    shown = frame.rename(
        columns={
            "endpoint": "Response endpoint",
            "input": "EEG input",
            "aggregation": "How values are combined",
            "quantity": "Calculated quantity",
            "n_patients": "Patients",
            "n_values": "Values",
            "responder_median": "Responder median",
            "nonresponder_median": "Non-responder median",
            "separation_auc": "Separation AUROC",
            "responder_direction": "Responder value",
            "patient_clustered_permutation_p": "Raw P",
            "holm_p_within_endpoint": "Holm-adjusted P",
            "significant_after_holm_0_05": "Significant after correction",
        }
    )
    columns = [
        "Response endpoint", "EEG input", "How values are combined",
        "Calculated quantity", "Patients", "Values",
        "Responder median", "Non-responder median",
        "Separation AUROC", "Responder value", "Raw P",
        "Holm-adjusted P", "Significant after correction",
    ]
    return (
        shown[columns]
        .style.hide(axis="index")
        .format(
            {
                "Responder median": "{:.3f}",
                "Non-responder median": "{:.3f}",
                "Separation AUROC": "{:.3f}",
                "Raw P": "{:.6g}",
                "Holm-adjusted P": "{:.6g}",
            }
        )
    )

## 1. One clip at a time

These rows answer the main question directly. Every value comes from one clip; no EEG values are averaged across clips. For PLI, each clip is divided into non-overlapping eight-second epochs within uninterrupted clean EEG. Raw delta-band PLI is averaged across epochs for every electrode pair, and the clip value is the percentage of pair means above 0.20. EEG on opposite sides of an artifact is never joined into an epoch.

In [4]:
single_clip = results.loc[results.analysis_family.eq("single_clip")]
display(readable_table(single_clip))

Response endpoint,EEG input,How values are combined,Calculated quantity,Patients,Values,Responder median,Non-responder median,Separation AUROC,Responder value,Raw P,Holm-adjusted P,Significant after correction
immediate,PRE awake,one clip,beta DFA intercept,50,100,-0.161,-0.094,0.566,lower,0.411921,0.411921,False
immediate,PRE awake,one clip,delta PLI connectivity,50,100,5.848,12.865,0.614,lower,0.154644,0.309288,False
immediate,PRE awake,one clip,R0 from that clip's DFA and PLI,50,100,-0.109,-0.667,0.642,higher,0.0722599,0.239652,False
immediate,POST sleep,one clip,beta Shannon entropy,50,100,5.918,5.385,0.781,higher,6.89999e-05,0.000414,True
immediate,POST awake,one clip,beta DFA intercept,50,100,-0.408,-0.164,0.689,lower,0.011153,0.0557649,False
sustained,PRE awake,one clip,beta DFA intercept,50,100,-0.211,0.017,0.655,lower,0.04167,0.0599439,False
sustained,PRE awake,one clip,delta PLI connectivity,50,100,5.263,12.865,0.666,lower,0.029972,0.0599439,False
sustained,PRE awake,one clip,R0 from that clip's DFA and PLI,50,100,-0.002,-0.749,0.725,higher,0.002757,0.010168,True
sustained,POST sleep,one clip,beta Shannon entropy,50,100,5.988,5.413,0.803,higher,1.4e-05,8.39999e-05,True
sustained,POST awake,one clip,beta DFA intercept,50,100,-0.440,-0.096,0.769,lower,0.000119,0.000594999,True


## 2. Combining POST awake and sleep

The published POST score R1 needs two different measurements: DFA from awake EEG and entropy from sleep EEG. The first row for each endpoint combines one awake clip with one sleep clip and includes all four possible within-patient pairings. The second row first averages the patient's two awake and two sleep clips, matching the source paper's patient-level construction.

In [5]:
post_combined = results.loc[
    results.quantity.str.contains("R1", case=False)
]
display(readable_table(post_combined))

Response endpoint,EEG input,How values are combined,Calculated quantity,Patients,Values,Responder median,Non-responder median,Separation AUROC,Responder value,Raw P,Holm-adjusted P,Significant after correction
immediate,POST awake + POST sleep,one awake clip paired with one sleep clip; all four within-patient pairs,R1 from awake DFA and sleep entropy,50,200,31.323,26.155,0.798,higher,4.9e-05,0.000343,True
immediate,POST awake + POST sleep,mean of two awake and two sleep clips,R1 after within-patient averaging,50,50,31.326,27.077,0.847,higher,2.9e-05,0.000232,True
sustained,POST awake + POST sleep,one awake clip paired with one sleep clip; all four within-patient pairs,R1 from awake DFA and sleep entropy,50,200,31.948,26.102,0.873,higher,2e-06,1.4e-05,True
sustained,POST awake + POST sleep,mean of two awake and two sleep clips,R1 after within-patient averaging,50,50,31.777,26.936,0.924,higher,9.99999e-07,7.99999e-06,True


## 3. PRE patient averaging for reference

The source PRE score has no sleep term. These two rows show what happens when the two PRE awake clip scores are averaged for each patient.

In [6]:
pre_average = results.loc[
    results.quantity.eq("R0 after within-patient averaging")
]
display(readable_table(pre_average))

Response endpoint,EEG input,How values are combined,Calculated quantity,Patients,Values,Responder median,Non-responder median,Separation AUROC,Responder value,Raw P,Holm-adjusted P,Significant after correction
immediate,PRE awake,mean of two awake clips,R0 after within-patient averaging,50,50,0.093,-0.607,0.661,higher,0.0599129,0.239652,False
sustained,PRE awake,mean of two awake clips,R0 after within-patient averaging,50,50,0.174,-0.841,0.747,higher,0.002542,0.010168,True


## Conclusions

- **Immediate response:** no PRE quantity is significant after correction. A single POST sleep entropy value is significant. The combined POST awake+sleep R1 is significant both before and after patient averaging.
- **Sustained response:** the single-clip PRE R0 is significant, although its two components considered separately are not significant after correction. Both single-clip POST components are significant. The combined POST awake+sleep R1 is strongly significant.
- These are same-cohort associations. The published quantities were selected using this cohort, so the table shows that response-related variation is present in these clips; it is not an independent validation of generalization to a new cohort.